# 2.BPE 分词器

## 2.1 unicode 标准

In [1]:
ord('s')

115

In [2]:
ord('牛')

29275

In [3]:
chr(115)

's'

In [4]:
chr(29275)

'牛'

#### problem：理解unicode

1.chr(0)返回的什么unicode字符

In [5]:
chr(0)

'\x00'

2.这个字符的字符串表示（__repr__()）与它的打印表示有什么不同

In [6]:
print(chr(0))

 


3.这个字符在文本中是什么样子

In [7]:
"this is a test" + chr(0) + "string"

'this is a test\x00string'

In [8]:
print("this is a test"+ chr(0) + "string")

this is a test string


## 2.2 Unicode 编码

In [9]:
test_string="hello! 孙嘉晨!"
utf8_encoded = test_string.encode("utf-8")

In [10]:
print(utf8_encoded)

b'hello! \xe5\xad\x99\xe5\x98\x89\xe6\x99\xa8!'


In [11]:
#得到encoded string的byte value （0-255的整数）
list(utf8_encoded)

[104,
 101,
 108,
 108,
 111,
 33,
 32,
 229,
 173,
 153,
 229,
 152,
 137,
 230,
 153,
 168,
 33]

In [12]:
#一个字节并不一定对应一个字符
print(len(test_string))
print(len(utf8_encoded))

11
17


In [13]:
print(utf8_encoded.decode("utf-8"))

hello! 孙嘉晨!


#### problem:Unicode 编码

1.相比于 UTF-16 或 UTF-32，为什么我们更倾向于在 UTF-8 编码的字节上训练 tokenizer？比较不同输入字符串在这几种编码下的输出可能会有帮助。

In [77]:
#UTF-8是可变长编码，对ASCII字符只使用1个字节，空间利用率高且兼容广泛。UTF-16与UTF-32使用固定或者更长的字节，导致词汇表更大、数据稀疏，不利于tokenzier学习高效子词单元
text="hello! 孙嘉晨!"
print(text.encode("utf-8"))
print(text.encode("utf-16"))
print(text.encode("utf-32"))

b'hello! \xe5\xad\x99\xe5\x98\x89\xe6\x99\xa8!'
b'\xff\xfeh\x00e\x00l\x00l\x00o\x00!\x00 \x00Y[\tVhf!\x00'
b'\xff\xfe\x00\x00h\x00\x00\x00e\x00\x00\x00l\x00\x00\x00l\x00\x00\x00o\x00\x00\x00!\x00\x00\x00 \x00\x00\x00Y[\x00\x00\tV\x00\x00hf\x00\x00!\x00\x00\x00'


2.考虑下面这个（错误的）函数，其目的是将UTF-8字节串解码为Unicode字符串。为什么这个函数是错误的？提供一个会产生错误结果的输入字节串的例子

In [78]:
def decode_utf8_bytes_to_str_wrong(bytestring:bytes):
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])

In [79]:
def decode_utf8_bytes_to_str_wrong(bytestring:bytes):#可以成功的
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])
decode_utf8_bytes_to_str_wrong("hello".encode("utf-8"))

'hello'

In [80]:
decode_utf8_bytes_to_str_wrong("café".encode("utf-8"))#不可以成功的

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc3 in position 0: unexpected end of data

3.给一个不解码成任何Unicode字符的双字节序列。

In [ ]:
b'\x80\x80'.decode('utf-8')

UnicodeDecodeError: 'utf-8' codec can't decode byte 0x80 in position 0: invalid start byte

## 2.3 subword Tokenization

一种介于word-level 跟byte-level 之间折中方案

## 2.4 BPE 分词器 训练

训练分为三部分：词表初始化、预分词、计算 BPE merges

In [19]:
pip install regex


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [20]:
import regex as re

In [21]:
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
re.findall(PAT,"some text that i'll pre-tokenize")

['some', ' text', ' that', ' i', "'ll", ' pre', '-', 'tokenize']

In [22]:
max([("A","B"),("A","C"),("B","ZZ"),("BA","A")])

('BA', 'A')

In [23]:
print("="*60)
print("BPE 分词器学习step-by-step ")
print("="*60)

BPE 分词器学习step-by-step 


In [24]:
print("lesson1 理解字节和字符串")
print("-"*40)

lesson1 理解字节和字符串
----------------------------------------


In [25]:
text="hello"
print(f"原始字符串:{text}")
print(f"字符串类型:{type(text)}")

原始字符串:hello
字符串类型:<class 'str'>


In [26]:
#转化为字节
bytes_text=text.encode("utf-8")
print(f"\n编码为字节:{bytes_text}")
print(f"字节类型:{type(bytes_text)}")


编码为字节:b'hello'
字节类型:<class 'bytes'>


In [27]:
#拆分为单个字节
byte_list=[bytes([b]) for b in bytes_text]
print(f"\n拆分成单字节列表:")
for i ,b in enumerate(byte_list):
    print(f" 位置{i}:{b}(值:{ord(b.decode())})")


拆分成单字节列表:
 位置0:b'h'(值:104)
 位置1:b'e'(值:101)
 位置2:b'l'(值:108)
 位置3:b'l'(值:108)
 位置4:b'o'(值:111)


In [28]:
print("\n 关键概念:")
print("- 字符串'hello' -> 字节 b'hello'")
print("- 每个字符对应1个字节（英文）")



 关键概念:
- 字符串'hello' -> 字节 b'hello'
- 每个字符对应1个字节（英文）


In [29]:
print("\n lesson2 统计相邻字节对")
print("-"*40)


 lesson2 统计相邻字节对
----------------------------------------


In [30]:
#准备一个简单的语料
corpus = ["hello","hello","world"]
print(f"语料:{corpus}")

语料:['hello', 'hello', 'world']


In [31]:
#统计pair 频率
from collections import Counter

In [32]:
all_pairs = []
for word in corpus:
    word_bytes = [bytes([b]) for b in word.encode("utf-8")]
    pairs = [(word_bytes[i],word_bytes[i+1])for i in range(len(word_bytes)-1)]
    all_pairs.extend(pairs)

In [33]:
pair_counts= Counter(all_pairs)
print(f"\n所有相邻对及频率:")
for pair ,count in pair_counts.most_common():
    print(f"{pair[0].decode()}+{pair[1].decode()}:{count}次")


所有相邻对及频率:
h+e:2次
e+l:2次
l+l:2次
l+o:2次
w+o:1次
o+r:1次
r+l:1次
l+d:1次


In [34]:
print("BPE 核心思想:")
print(f" -最频繁的对是:{pair_counts.most_common(1)[0][0]}")
print(" -我们应该先合并这个！")

BPE 核心思想:
 -最频繁的对是:(b'h', b'e')
 -我们应该先合并这个！


In [35]:
print("\n\n lesson3 手动模拟一次BPE 合并")
print("-"*40)



 lesson3 手动模拟一次BPE 合并
----------------------------------------


In [36]:
#初始状态
vocab = {i:bytes([i])for i in range(256)}
sequences = [
    tuple(bytes([b]) for b in "hello".encode("utf-8")),
    tuple(bytes([b]) for b in "world".encode("utf-8")),
]

In [37]:
print(f"初始词汇表大小:{len(vocab)}")
print(f"初始序列：")
for seq in sequences:
    print(f" {seq}")

初始词汇表大小:256
初始序列：
 (b'h', b'e', b'l', b'l', b'o')
 (b'w', b'o', b'r', b'l', b'd')


In [38]:
print("\n--- 第1次合并 ---")


--- 第1次合并 ---


In [39]:
#统计所pairs
pair_counts = Counter()
for seq in sequences:
    for i in range(len(seq)-1):
        pair = (seq[i],seq[i+1])
        pair_counts[pair]+=1

In [40]:
print("Pair 频率统计:")
for pair,count in pair_counts.most_common(5):
    print(f" {pair[0].decode()}+{pair[1].decode()}:{count}次")


Pair 频率统计:
 h+e:1次
 e+l:1次
 l+l:1次
 l+o:1次
 w+o:1次


In [41]:
#选择最频繁的pair
best_pair = max(pair_counts,key=pair_counts.get)
print(f" 选择最频繁的 pair:{best_pair[0].decode()}+{best_pair[1].decode()}")

 选择最频繁的 pair:h+e


In [42]:
#创建新的token
new_token = best_pair[0]+best_pair[1]
new_id = len(vocab)
vocab[new_id]=new_token

In [43]:
print(f"创建新 token : ID ={new_id},值={new_token}")

创建新 token : ID =256,值=b'he'


In [44]:
#在序列中合并
def merge_in_sequences(seq,pair,new_token):
    new_seq=[]
    i=0
    while i<len(seq):
        if i < len(seq)-1 and seq[i] == pair[0] and seq[i+1] == pair[1]:
            new_seq.append(new_token)
            i+=2
        else:
            new_seq.append(seq[i])
            i+=1
    return tuple(new_seq)

In [45]:
sequences=[merge_in_sequences(seq,best_pair,new_token) for seq in sequences]

In [46]:
print(f"n 合并后序列:")
for seq in sequences:
    print(f"{seq}")

n 合并后序列:
(b'he', b'l', b'l', b'o')
(b'w', b'o', b'r', b'l', b'd')


In [47]:
print(f"词汇表现大小:{len(vocab)}")

词汇表现大小:257


In [48]:
print("lesson4 完整 BPE 训练循环")
print("-"*40)

lesson4 完整 BPE 训练循环
----------------------------------------


In [49]:
#重新初始化
vocab = {i:bytes([i]) for i in range(256)}
sequences = [
    tuple(bytes([b]) for b in word.encode("utf-8"))
    for word in ["hello","hello","world","hell","hello"]
]

In [50]:
sequences

[(b'h', b'e', b'l', b'l', b'o'),
 (b'h', b'e', b'l', b'l', b'o'),
 (b'w', b'o', b'r', b'l', b'd'),
 (b'h', b'e', b'l', b'l'),
 (b'h', b'e', b'l', b'l', b'o')]

In [51]:
target_vocab_size=260
merges=[]

In [52]:
print(f"目标词汇量大小:{target_vocab_size}")
print(f"初始词汇表:{len(vocab)}个tokens")

目标词汇量大小:260
初始词汇表:256个tokens


In [53]:
iteration=0
while len(vocab) < target_vocab_size:
    iteration+=1

    #统计pairs频率
    pair_counts=Counter()
    for seq in sequences:
        for i in range(len(seq)-1):
            pair=(seq[i],seq[i+1])
            pair_counts[pair]+=1
    if not pair_counts:
        break

    #选择最频繁的pair
    best_pair = max(pair_counts,key = pair_counts.get)
    max_count=pair_counts[best_pair]

    #创建新的token
    new_token=best_pair[0]+best_pair[1]
    new_id =len(vocab)
    vocab[new_id]=new_token

    #记录合并
    merges.append(best_pair)

    #在序列中合并
    sequences=[merge_in_sequences(seq,best_pair,new_token)for seq in sequences]

    print(f"迭代:{iteration}:")
    print(f"合并:{best_pair[0].decode()}+{best_pair[1].decode()}->{new_token.decode()}")
    print(f"频率：{max_count}次")
    print(f"新 ID:{new_id}")
    print(f"词汇表大小:{len(vocab)}")


迭代:1:
合并:h+e->he
频率：4次
新 ID:256
词汇表大小:257
迭代:2:
合并:he+l->hel
频率：4次
新 ID:257
词汇表大小:258
迭代:3:
合并:hel+l->hell
频率：4次
新 ID:258
词汇表大小:259
迭代:4:
合并:hell+o->hello
频率：3次
新 ID:259
词汇表大小:260


In [54]:
print("\n"+"="*60)
print("最终结果:")
print(f"词汇表大小:{len(vocab)}")
print(f"\n 合并规则:")
for i,(p1,p2) in enumerate(merges):
    print(f" {i+1}.{p1.decode()}+{p2.decode()}->{(p1+p2).decode()}")



最终结果:
词汇表大小:260

 合并规则:
 1.h+e->he
 2.he+l->hel
 3.hel+l->hell
 4.hell+o->hello


In [55]:
print("最终序列:")
for seq in sequences:
    decoded=b''.join(seq).decode("utf-8")
    print(f"{list(seq)}->'{decoded}'")

最终序列:
[b'hello']->'hello'
[b'hello']->'hello'
[b'w', b'o', b'r', b'l', b'd']->'world'
[b'hell']->'hell'
[b'hello']->'hello'


In [56]:
sequences

[(b'hello',),
 (b'hello',),
 (b'w', b'o', b'r', b'l', b'd'),
 (b'hell',),
 (b'hello',)]

In [57]:
print("lesson5 为什么要用字节级别?")
print("-"*40)

lesson5 为什么要用字节级别?
----------------------------------------


In [58]:
print("Q1:中文字符怎么办？")
chinese="你好"
chinese_bytes = chinese.encode("utf-8")
print(f" '{chinese}'->{chinese_bytes}")
print(f"  占用{len(chinese_bytes)}字节")


Q1:中文字符怎么办？
 '你好'->b'\xe4\xbd\xa0\xe5\xa5\xbd'
  占用6字节


In [59]:
print("Q2:emoji怎么办?")
emoji="😅"
emoji_bytes=emoji.encode("utf-8")
print(f" '{emoji}'->{emoji_bytes}")
print(f"  占用{len(emoji_bytes)}字节")


Q2:emoji怎么办?
 '😅'->b'\xf0\x9f\x98\x85'
  占用4字节


In [60]:
print(" 字节级别的优势:")
print("  - 统一处理所有语言")
print("  - 不需要预先知道词汇表")
print("  - 可以处理任意 Unicode 字符")

 字节级别的优势:
  - 统一处理所有语言
  - 不需要预先知道词汇表
  - 可以处理任意 Unicode 字符


In [61]:
print("lesson6 理解增量更新优化")
print("-"*40)

lesson6 理解增量更新优化
----------------------------------------


In [62]:
print("朴素方法(慢):每次合并都重新统计所有pair频率,时间复杂度O(merges * corpus_size)")

朴素方法(慢):每次合并都重新统计所有pair频率,时间复杂度O(merges * corpus_size)


In [63]:
print("优化方法(快):只更新受影响的pairs,例如合并(e,r)->er")
print(" -删除:(h,e),(e,r),(r,ing)")
print(" -添加:(h,er),(er,ing)")
print(" -其他pairs不变")


优化方法(快):只更新受影响的pairs,例如合并(e,r)->er
 -删除:(h,e),(e,r),(r,ing)
 -添加:(h,er),(er,ing)
 -其他pairs不变


#### Problem: (ecoding) BPE Tokenizer Training

In [64]:
import os
import re
from collections import defaultdict
from typing import List,Tuple,Dict
import regex

In [65]:
#GPT-2 预分词
GPT2_PATTERN = regex.compile(
    r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    )


In [68]:
def run_train_bpe(
    input_path:str | os.PathLike,
    vocab_size:int,
    special_tokens:list[str],
    **kwargs,
)-> tuple[dict[int,bytes],list[tuple[bytes,bytes]]]:
    #step1:参数校验
    if not isinstance(vocab_size,int) or vocab_size<=0:
        raise ValueError("vocab_size 必须是正整数")
    #step2初始化词汇表
    vocab: Dict[int,bytes]= {i:bytes([i]) for i in range(256)}
    next_token_id :int =256

    existing_bytes_values=set(vocab.values())

    for st_str in special_tokens:
        if len(vocab)>= vocab_size:
            break
        st_bytes=st_str.encode("utf-8")
        if st_bytes not in existing_bytes_values:
            vocab[next_token_id]=st_bytes
            existing_bytes_values.add(st_bytes)
            next_token_id+=1
    #step3:读取训练预料
    try :
        with open(input_path,"r",encoding="utf-8",errors="ignore") as f:
            text = f.read()
    except FileNotFoundError:
        text="" #文件不存在时视为空文本
    #step4:预处理和预分词
    ##4.1 按特殊tokens分割
    if special_tokens:
        split_pattern = "|".join(re.escape(tok) for tok in special_tokens)
        chunks = regex.split(split_pattern,text)
    else:
        chunks=[text]
    ##4.2 对每个chunk进行预分词
    token_frequency_table = defaultdict(int)
    for chunk in chunks:
        for word in regex.findall(GPT2_PATTERN,chunk):
            word_bytes = word.encode("utf-8")
            byte_tuple=tuple(bytes([b]) for b in word_bytes)
            token_frequency_table[byte_tuple]+=1
    #step5:初始化 pair 频率统计
    pair_counts=defaultdict(int)
    for byte_tuple,freq in token_frequency_table.items():
        for i in range(len(byte_tuple)-1):
            pair=(byte_tuple[i],byte_tuple[i+1])
            pair_counts[pair]+=freq
    #step6: BPE迭代合并
    merges:List[Tuple[bytes,bytes]]=[]
    while len(vocab)<vocab_size:
        #检查终止条件:
        if not pair_counts:
            break
        #选择最频繁的pair
        max_count=max(pair_counts.values())
        candidates=[pair for pair,count in pair_counts.items() if count == max_count]
        best_pair=max(candidates)
        #创建新token
        new_token_bytes=best_pair[0]+best_pair[1]
        vocab[next_token_id]=new_token_bytes
        next_token_id+=1
        merges.append(best_pair)
        #增量更新pair_counts(性能优化的关键)
        affected_tokens=[]
        for byte_tuple,freq in token_frequency_table.items():
            has_pair = False
            for i in range(len(byte_tuple)-1):
                if byte_tuple[i] == best_pair[0] and byte_tuple[i+1] == best_pair[1]:
                    has_pair = True
                    break
            if has_pair:
                affected_tokens.append((byte_tuple,freq))
        for byte_tuple,freq in affected_tokens:
             for i in range(len(byte_tuple)-1):
                old_pair = (byte_tuple[i],byte_tuple[i+1])
                pair_counts[old_pair]-= freq
                if pair_counts[old_pair] <=0:                        
                    del pair_counts[old_pair]
             new_byte_tuple = _merge_pair_in_sequence(byte_tuple,best_pair,new_token_bytes)
             for i in range(len(new_byte_tuple)-1):
                new_pair=(new_byte_tuple[i],new_byte_tuple[i+1])
                pair_counts[new_pair]+= freq
             del token_frequency_table[byte_tuple]
             token_frequency_table[new_byte_tuple]+= freq
    return vocab,merges


def _merge_pair_in_sequence(
    byte_sequence:tuple[bytes,...],
    pair:tuple[bytes,bytes],
    new_token:bytes,
)-> tuple[bytes,...]:
    new_sequence=[]
    i=0
    while i < len(byte_sequence):
        if i < len(byte_sequence)-1 and byte_sequence[i] == pair[0] and byte_sequence[i+1] == pair[1]:
            new_sequence.append(new_token)
            i+=2
        else:
            new_sequence.append(byte_sequence[i])
            i+=1
    return tuple(new_sequence)

def save_vocab_and_merges(
        vocab:dict[int,bytes],
        merges:list[tuple[bytes,bytes]],
        vocab_path:str,
        merges_path:str,
):  
    import json
    vocab_str={
        token_id:token_bytes.decode("utf-8",errors="replace")
        for token_id,token_bytes in vocab.items()
    }
    with open(vocab_path,"w",encoding="utf-8") as f:
        json.dump(vocab_str, f, ensure_ascii=False, indent=2)
    with open(merges_path,"w",encoding="utf-8") as f:
        for p1,p2 in merges:
            p1_str=p1.decode("utf-8",errors="replace")
            p2_str=p2.decode("utf-8",errors="replace")
            f.write(f"{p1_str} {p2_str}\n")



#### Problem: BPE training on TinyStories

In [2]:
%cd /Users/apple/Desktop/cs336/assignment1-basics-main/homework

/Users/apple/Desktop/cs336/assignment1-basics-main/homework


In [ ]:
from BpeTrainingOnTinystories import main
main()

BPE Training on TinyStories Dataset
输入文件: /Users/apple/Desktop/cs336/assignment1-basics-main/data/TinyStories-train.txt
目标词汇表大小: 10,000
特殊tokens: ['<|endoftext|>']

📊 输入文件信息:
   文件大小: 1.79 GB

🚀 开始训练...


1.在TinyStories 数据集上训练一个字节级别的 BPE 分词器，使用最大词汇表大小10,000。确保将 TinyStories 的特殊 token <｜end▁of▁sentence｜> 添加到词汇表中。将生成的词汇表和合并规则序列化到磁盘以供后续检查。
问：1）训练花了多少小时和内存？2）词汇表中最长的token是什么？3）他合理吗？

In [ ]:
from BpeTrainingOnTinystories import performance_analysis
performance_analysis()


🔬 性能分析模式

✅ 性能分析训练完成:
   词汇表大小: 10,000
   合并次数: 9,743

📊 性能分析结果 (按累计时间排序, 前30名):
----------------------------------------------------------------------
         3523502191 function calls (3523502183 primitive calls) in 713.342 seconds

   Ordered by: cumulative time
   List reduced from 214 to 30 due to restriction <30>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      143    0.001    0.000 1024.433    7.164 /opt/miniconda3/lib/python3.13/asyncio/base_events.py:1970(_run_once)
      143  106.780    0.747  706.946    4.944 /opt/miniconda3/lib/python3.13/selectors.py:540(select)
      142  372.238    2.621  524.317    3.692 {method 'control' of 'select.kqueue' objects}
2387878257  155.414    0.000  155.414    0.000 /Users/apple/Desktop/cs336/assignment1-basics-main/homework/bpe_tokenizer.py:131(<genexpr>)
463596707   31.170    0.000   31.170    0.000 {method 'encode' of 'str' objects}
669750128   24.393    0.000   24.393    0.000 {built-in method builtins.le

1）402.43s;103.69MB。2）<｜end▁of▁sentence｜> 。3）特殊单词总长度为20，日常少见过超过20个字母的单词，所以特殊字符最长比较合理

2.分析你的代码。分词器训练过程中哪个部分耗时最多

异步I/O 开销

#### problem:: BPE Training on OpenWebText

1.在 OpenWebText 数据集上训练一个字节级别的 BPE 分词器，最大词汇表大小为 32,000。将生成的词汇表和合并规则序列化到磁盘以供后续检查。
问：1）词汇表中最长的 token 是什么？这合理吗？2）比较和对比在 TinyStories 与 OpenWebText 上训练得到的分词器